# Story-to-Music — flan-t5-small Fine-Tuning
**Kaggle T4 GPU** üzerinde çalışır.

Başlamadan önce:
1. Sağ panel → **Add Data** → **Upload** → `dataset.jsonl` dosyasını yükle
2. Accelerator: **GPU T4 x2** seçili olsun
3. Hücreleri sırayla çalıştır

In [ ]:
# ── 1. Kurulum ────────────────────────────────────────────────────────────────
!pip install -q transformers datasets accelerate sentencepiece protobuf
print('Kurulum tamamlandı.')

In [ ]:
# ── 2. Dataset Konumu ─────────────────────────────────────────────────────────
import glob, os

# Kaggle'da Add Data ile yüklenen dosyalar /kaggle/input/ altına gelir
candidates = glob.glob('/kaggle/input/**/dataset.jsonl', recursive=True)

if candidates:
    DATASET_PATH = candidates[0]
    print(f'Dataset bulundu: {DATASET_PATH}')
else:
    # Notebook dizinine manuel kopyaladıysan
    DATASET_PATH = '/kaggle/working/dataset.jsonl'
    print(f'[!] /kaggle/input/ altında bulunamadı. Add Data ile yükle.')

# Kayıt sayısını kontrol et
with open(DATASET_PATH, encoding='utf-8') as f:
    n = sum(1 for l in f if l.strip())
print(f'Toplam kayıt: {n}')

In [ ]:
# ── 3. Konfigürasyon ──────────────────────────────────────────────────────────
BASE_MODEL     = 'google/flan-t5-small'
OUTPUT_DIR     = '/kaggle/working/story-to-music-t5'
CHECKPOINT_DIR = '/kaggle/working/checkpoints'

MAX_INPUT_LEN  = 512
MAX_TARGET_LEN = 512
TRAIN_RATIO    = 0.90

EPOCHS         = 15
BATCH_SIZE     = 8      # OOM olursa 4'e düşür
GRAD_ACCUM     = 2      # efektif batch = 16
LR             = 5e-4
WARMUP_RATIO   = 0.10
WEIGHT_DECAY   = 0.01
PATIENCE       = 3

print('Konfigürasyon tamam.')

In [ ]:
# ── 4. Veri Yükleme ───────────────────────────────────────────────────────────
import json
from datasets import Dataset

records = []
with open(DATASET_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        d = json.loads(line)
        if 'input_text' in d and 'target_text' in d:
            records.append({'input_text': d['input_text'], 'target_text': d['target_text']})

ds = Dataset.from_list(records)
split = ds.train_test_split(test_size=1 - TRAIN_RATIO, seed=42)
train_ds, eval_ds = split['train'], split['test']

print(f'Eğitim: {len(train_ds)} | Doğrulama: {len(eval_ds)}')

In [ ]:
# ── 5. Model ve Tokenizer ─────────────────────────────────────────────────────
import torch
from transformers import T5ForConditionalGeneration, T5TokenizerFast

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Cihaz: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

tokenizer = T5TokenizerFast.from_pretrained(BASE_MODEL)
model = T5ForConditionalGeneration.from_pretrained(BASE_MODEL)
print(f'Parametre sayısı: {model.num_parameters():,}')

In [ ]:
# ── 6. Tokenizasyon ───────────────────────────────────────────────────────────
from transformers import DataCollatorForSeq2Seq

def tokenize(batch):
    model_inputs = tokenizer(
        batch['input_text'],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=batch['target_text'],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False,
    )
    model_inputs['labels'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in ids]
        for ids in labels['input_ids']
    ]
    return model_inputs

train_tok = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
eval_tok  = eval_ds.map(tokenize,  batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8
)

print('Tokenizasyon tamam.')

In [ ]:
# ── 7. Eğitim ─────────────────────────────────────────────────────────────────
from pathlib import Path
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)

total_steps  = (len(train_tok) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,

    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type='cosine',

    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=3,

    fp16=(device == 'cuda'),
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,

    logging_steps=20,
    report_to='none',
    seed=42,
)

# transformers >= 4.46: 'tokenizer' parametresi 'processing_class' olarak değişti
import transformers
trainer_kwargs = dict(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)
if tuple(int(x) for x in transformers.__version__.split('.')[:2]) >= (4, 46):
    trainer_kwargs['processing_class'] = tokenizer
else:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

# Önceki checkpoint varsa devam et
checkpoints = sorted(Path(CHECKPOINT_DIR).glob('checkpoint-*'), key=lambda x: int(x.name.split('-')[-1])) if Path(CHECKPOINT_DIR).exists() else []
last_ckpt = str(checkpoints[-1]) if checkpoints else None
if last_ckpt:
    print(f'Checkpoint bulundu, devam ediliyor: {last_ckpt}')

trainer.train(resume_from_checkpoint=last_ckpt)

In [ ]:
# ── 8. Modeli Kaydet ──────────────────────────────────────────────────────────
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Boyut raporu
total_mb = sum(f.stat().st_size for f in Path(OUTPUT_DIR).rglob('*') if f.is_file()) / (1024**2)
print(f'Model kaydedildi: {OUTPUT_DIR}')
print(f'Toplam boyut: {total_mb:.1f} MB')

In [ ]:
# ── 9. Smoke Test ─────────────────────────────────────────────────────────────
import textwrap

ornek = eval_ds[0]['input_text']
print('GİRDİ (ilk 200 karakter):')
print(textwrap.shorten(ornek, 200))
print()

model.eval()
inputs = tokenizer(ornek, return_tensors='pt', max_length=MAX_INPUT_LEN, truncation=True).to(device)
with torch.no_grad():
    out = model.generate(**inputs, max_length=256, num_beams=4, early_stopping=True)

decoded = tokenizer.decode(out[0], skip_special_tokens=True)
print('ÇIKTI:')
print(decoded[:500])

In [ ]:
# ── 10. Değerlendirme ─────────────────────────────────────────────────────────
import random

REQUIRED_KEYS = {'emotion','energy','bpm','key','instruments','vocal_style','suno_style_prompt','structured_lyrics'}
N_EVAL = 50

samples = random.sample(list(range(len(eval_ds))), min(N_EVAL, len(eval_ds)))
valid_json, coverages = 0, []

for idx in samples:
    inp = eval_ds[idx]['input_text']
    ref = eval_ds[idx]['target_text']

    inputs = tokenizer(inp, return_tensors='pt', max_length=MAX_INPUT_LEN, truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_length=MAX_TARGET_LEN, num_beams=4, early_stopping=True)
    gen = tokenizer.decode(out[0], skip_special_tokens=True)

    try:
        parsed = json.loads(gen)
        valid_json += 1
        coverage = len(REQUIRED_KEYS & set(parsed.keys())) / len(REQUIRED_KEYS)
    except:
        coverage = 0.0
    coverages.append(coverage)

n = len(samples)
avg_cov = sum(coverages) / n
json_rate = valid_json / n

print(f'Geçerli JSON  : {valid_json}/{n} ({json_rate:.0%})')
print(f'Alan kapsamı  : {avg_cov:.0%}')

if json_rate >= 0.80 and avg_cov >= 0.85:
    print('\n✓ Model sunucu entegrasyonuna hazır.')
else:
    print('\n[!] Henüz yeterli değil — daha fazla epoch veya veri artırımı dene.')

In [ ]:
# ── 11. Modeli İndir (zip) ────────────────────────────────────────────────────
import shutil

zip_path = '/kaggle/working/story-to-music-t5'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f'İndirilmeye hazır: {zip_path}.zip')
print('Kaggle sağ paneli → Output → story-to-music-t5.zip → Download')